In [1]:
quiet_library <- function(...) { suppressWarnings(suppressPackageStartupMessages(library(...))) }
quiet_library(hise)
quiet_library(ArchR)
quiet_library(purrr)
quiet_library(dplyr)


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .______      
          /   \     |   _ 

In [2]:
out_dir = "output/tcell-vrd_bigwigs"
if(!dir.exists(out_dir)) {
    dir.create(out_dir, recursive = TRUE)
}

In [3]:
sample_manifest <- read.csv("../common/EXP00454_TEAseq_sample_manifest.csv")

In [4]:
sample_meta <- sample_manifest %>%
  select(pbmc_sample_id, treatment, timepoint) %>%
  mutate(Sample = paste0("EXP-00454-P1_", pbmc_sample_id),
         treat_time = paste0(treatment, "_", timepoint)) %>%
  select(-pbmc_sample_id)
head(sample_meta)

,treatment,timepoint,Sample,treat_time
,<chr>,<int>,<chr>,<chr>
1,lenalidomide,72,EXP-00454-P1_PC02184-038,lenalidomide_72
2,bortezomib,72,EXP-00454-P1_PC02184-039,bortezomib_72
3,dmso,72,EXP-00454-P1_PC02184-040,dmso_72
4,dexamethasone,24,EXP-00454-P1_PC02184-041,dexamethasone_24
5,lenalidomide,24,EXP-00454-P1_PC02184-042,lenalidomide_24
6,bortezomib,24,EXP-00454-P1_PC02184-043,bortezomib_24


In [5]:
atac_file_uuids <- list(
    "a7f77e33-5a1b-4210-9e1c-100c458e4383", 
    "9d062f9a-4f51-4c2b-9e43-b49eb5e002a3", 
    "eda0b361-880d-4941-a9d4-5ebc85cdda01", 
    "163f6153-6db3-4957-8ec6-a12f5fdd29a7", 
    "560d3e7a-d9fd-424f-860c-cc60c4632f50", 
    "871c76aa-cee6-4f99-8faf-72e6b9210f56"
)

In [6]:
atac_tar_files <- hise::cacheFiles(
    atac_file_uuids
)

[2026-06-15 10:08:33] INFO  Calling hise::cacheFiles
[1] "downloading fileID a7f77e33-5a1b-4210-9e1c-100c458e4383"
[1] "downloading fileID 9d062f9a-4f51-4c2b-9e43-b49eb5e002a3"
[1] "downloading fileID eda0b361-880d-4941-a9d4-5ebc85cdda01"
[1] "downloading fileID 163f6153-6db3-4957-8ec6-a12f5fdd29a7"
[1] "downloading fileID 560d3e7a-d9fd-424f-860c-cc60c4632f50"
[1] "downloading fileID 871c76aa-cee6-4f99-8faf-72e6b9210f56"
[2026-06-15 10:08:48] INFO  Finished hise::cacheFiles (success=TRUE, time_elapsed=12.545s)


In [7]:
walk(
    atac_tar_files,
    function(tf) {
        command <- paste("tar -xf", tf)
        system(command)
    }
)

Note: un-tar-ing the files moves them to the `output` directory, based on their original filenames.

In [8]:
type_paths <- list.files(
    "output",
    full.names = TRUE
)
type_paths <- type_paths[grepl("vrdtea_ArchR", type_paths)]

In [9]:
cell_types <- sub(".+-(.+)_20.+", "\\1", type_paths)
cell_types

[1] "t_cd4_cm"     "t_cd4_em"     "t_cd4_naive"  "t_cd4_treg"   "t_cd8_memory"
[6] "t_cd8_naive"

In [10]:
type_proj <- map(
    type_paths,
    loadArchRProject,
    showLogo = FALSE
)
names(type_proj) <- cell_types

Successfully loaded ArchRProject!

Successfully loaded ArchRProject!

Successfully loaded ArchRProject!

Successfully loaded ArchRProject!

Successfully loaded ArchRProject!

Successfully loaded ArchRProject!



In [11]:
archr_bw_files <- map(
    type_proj,
    getGroupBW,
    groupBy = "Sample",
    tileSize = 10
)

ArchR logging to : ArchRLogs/ArchR-getGroupBW-7683535cb2a-Date-2026-06-15_Time-10-19-22.166908.log
If there is an issue, please report to github with logFile!

2026-06-15 10:19:32.906449 : EXP-00454-P1_PC02184-038 (1 of 12) : Creating BigWig for Group, 0.16 mins elapsed.

2026-06-15 10:21:51.367385 : EXP-00454-P1_PC02184-039 (2 of 12) : Creating BigWig for Group, 2.468 mins elapsed.

2026-06-15 10:23:36.444783 : EXP-00454-P1_PC02184-040 (3 of 12) : Creating BigWig for Group, 4.219 mins elapsed.

2026-06-15 10:25:20.649563 : EXP-00454-P1_PC02184-041 (4 of 12) : Creating BigWig for Group, 5.956 mins elapsed.

2026-06-15 10:27:10.574125 : EXP-00454-P1_PC02184-042 (5 of 12) : Creating BigWig for Group, 7.788 mins elapsed.

2026-06-15 10:29:08.972102 : EXP-00454-P1_PC02184-043 (6 of 12) : Creating BigWig for Group, 9.761 mins elapsed.

2026-06-15 10:30:58.577735 : EXP-00454-P1_PC02184-044 (7 of 12) : Creating BigWig for Group, 11.588 mins elapsed.

2026-06-15 10:32:52.16971 : EXP-00454-P1_P

In [12]:
fn_to_sample <- function(fn) {
    bn <- basename(fn)
    sample <- sub("-.+", "", bn)
    sample <- gsub("\\.", "-", sample)
    sample
}

In [13]:
tt <- sample_meta$treat_time[sample_meta$Sample == "EXP-00454-P1_PC02184-038"]

In [14]:
tt

[1] "lenalidomide_72"

In [15]:
test <- unlist(archr_bw_files)

In [16]:
out_bw_files <- map2(
    unlist(archr_bw_files),
    names(unlist(archr_bw_files)),
    function(fn, group_name) {
        sample <- fn_to_sample(fn)
        ct <- sub("[0-9]+$", "", group_name)
        ct <- gsub("_","-",ct)
        tt <- sample_meta$treat_time[sample_meta$Sample == sample]
        bw_out <- paste0(
            "tcell-vrd_",
            ct,"_",tt,
            "_archr.bw"
        )
        out_file <- file.path(out_dir, bw_out)
        file.copy(fn, out_file)
        out_file
    }
)

In [17]:
out_bw_tar <- paste0("output/tcell-vrd_cell-type_treat_time_bigwigs_", Sys.Date(),".tar")
system(paste("tar -cf", out_bw_tar, "output/tcell-vrd_bigwigs"))

## Build metadata sheet

In [29]:
x <- out_bw_files[[1]]
test <- function(x) strsplit(x, "_")[[1]][4]
test(x)

[1] "lenalidomide"

In [35]:
cell_type_update <- c(
    "t-cd4-cm" = "CD4 CM",
    "t-cd4-em" = "CD4 EM",
    "t-cd4-naive" = "CD4 Naive",
    "t-cd4-treg" = "CD4 Treg",
    "t-cd8-memory" = "CD8 Memory",
    "t-cd8-naive" = "CD8 Naive"
)
treatment_update <- c(
    "bortezomib" = "Bortezomib",
    "dexamethasone" = "Dexamethasone",
    "dmso" = "DMSO",
    "lenalidomide" = "Lenalidomide",
    "untreated" = "Untreated"
)
timepoint_update <- c(
    "0" = "T0",
    "4" = "T4",
    "24" = "T24",
    "72" = "T72"
)

In [38]:
bigwig_metadata <- data.frame(
    filename = unlist(out_bw_files),
    "Cell Type" = cell_type_update[map_chr(out_bw_files, function(x) strsplit(x, "_")[[1]][3])],
    "Treatment" = treatment_update[map_chr(out_bw_files, function(x) strsplit(x, "_")[[1]][4])],
    "Timepoint" = timepoint_update[map_chr(out_bw_files, function(x) strsplit(x, "_")[[1]][5])]
)

In [40]:
rownames(bigwig_metadata) <- NULL

In [42]:
out_bw_meta <- paste0("output/tcell-vrd_cell-type_treat_time_bigwigs_metadata_", Sys.Date(),".csv")
write.csv(bigwig_metadata, out_bw_meta, row.names = FALSE)

## Store results in HISE

In [43]:
study_space_uuid <- "40df6403-29f0-4b45-ab7d-f46d420c422e"
title <- "VRd TEA-seq scATAC BigWig files"

In [44]:
upload_id <- ids::adjective_animal()
upload_id

[1] "nonodorous_mite"

In [45]:
out_list <- as.list(c(out_bw_tar, out_bw_meta))

In [46]:
out_list

[[1]]
[1] "output/tcell-vrd_cell-type_treat_time_bigwigs_2026-06-15.tar"

[[2]]
[1] "output/tcell-vrd_cell-type_treat_time_bigwigs_metadata_2026-06-15.csv"

In [47]:
sessionInfo()

R version 4.4.1 (2024-06-14)
Platform: x86_64-conda-linux-gnu
Running under: Ubuntu 22.04.5 LTS

Matrix products: default
BLAS/LAPACK: /home/workspace/environment/archrpixiv11/.pixi/envs/default/lib/libopenblasp-r0.3.32.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=C.UTF-8    LC_NUMERIC=C        LC_TIME=C          
 [4] LC_COLLATE=C        LC_MONETARY=C       LC_MESSAGES=C      
 [7] LC_PAPER=C          LC_NAME=C           LC_ADDRESS=C       
[10] LC_TELEPHONE=C      LC_MEASUREMENT=C    LC_IDENTIFICATION=C

time zone: America/Los_Angeles
tzcode source: system (glibc)

attached base packages:
[1] stats4    grid      stats     graphics  grDevices utils     datasets 
[8] methods   base     

other attached packages:
 [1] dplyr_1.2.1                 purrr_1.2.1                
 [3] rhdf5_2.50.0                SummarizedExperiment_1.36.0
 [5] Biobase_2.66.0              RcppArmadillo_15.2.4-1     
 [7] Rcpp_1.1.1                  Matrix_1.7-5               
 [9] GenomicRanges_1.58.0    

In [49]:
uploadFiles(
    files = out_list,
    fileTypes = list(
        "tar#derived",
        "Generic CSV File#ProjectStore"
    ),
    studySpaceId = study_space_uuid,
    title = title,
    inputFileIds = unname(atac_file_uuids),
    destination = upload_id
)

Please provide input of comma separated sample ids for the files being uploaded. If you do not have any sample ids, press enter:  


[2026-06-15 13:26:34] INFO  Calling uploadFiles
[1] "Retrying..."
[2026-06-15 13:27:02] INFO  Finished uploadFiles (success=TRUE, time_elapsed=26.498s)


$Message
[1] "General Okay-ness"

$VisualizationId
[1] "00000000-0000-0000-0000-000000000000"

$AbstractionId
[1] "00000000-0000-0000-0000-000000000000"

$TraceId
[1] "9b934048-0caa-48ca-ba9d-67fd4578753e"

$ProcessId
[1] "6258e1d0-8d2b-4732-afb0-3ca733ecf0e9"

$WorkflowId
[1] "f4c0e907-0711-402e-93d7-17a5762bb6af"

$FileIds
$FileIds[[1]]
[1] "38f4ed62-7799-43fb-adac-0e9816cdba15"

$FileIds[[2]]
[1] "1a441e53-6b37-4578-a03f-a3260a19be95"